# CoRe-TFM — JMLR Robustness EXECUTABLE V2.2

Use this notebook instead of V2/V2.1. V2.2 fixes the false IPython-syntax preflight and starts from a clean evidence root so an older frozen orchestration commit cannot be resumed accidentally.

One run works on the next incomplete real-TFM variant. Re-run **Run all** after a disconnect or after a variant finishes. Fold checkpoints persist in Drive.


## 1. Setup, regression-test the runner, and freeze the source

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, traceback
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/core-tfm')
ROBUST_ROOT = Path('/content/drive/MyDrive/CoRe_TFM_Q1/core_tfm_jmlr_robustness_v2_2')
ROBUST_ROOT.mkdir(parents=True, exist_ok=True)
PROTOCOL_PATH = ROBUST_ROOT/'ROBUSTNESS_PROTOCOL.json'

if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','https://github.com/bnssaanirudh/core-tfm.git',str(ROOT)], check=True)
subprocess.run(['git','fetch','origin'], cwd=ROOT, check=True)

if PROTOCOL_PATH.exists():
    existing_protocol = json.loads(PROTOCOL_PATH.read_text())
    SOURCE_COMMIT = existing_protocol['source_commit_at_freeze']
    print('Resuming frozen V2.2 source:', SOURCE_COMMIT)
else:
    SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','origin/main'], cwd=ROOT, text=True).strip()
    print('Candidate source commit:', SOURCE_COMMIT)

subprocess.run(['git','checkout','--detach',SOURCE_COMMIT], cwd=ROOT, check=True)
HEAD = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
assert HEAD == SOURCE_COMMIT

subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[test]','pyyaml','ucimlrepo','hf_transfer'], cwd=ROOT, check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_robustness_runner.py'], cwd=ROOT, check=True)

SRC = str(ROOT/'src')
if SRC not in sys.path: sys.path.insert(0, SRC)
os.chdir(ROOT)

from core_tfm.robustness_runner import load_notebook, patch_q1_notebook, code_cells_through_shard_12e, fold_result_status, write_complete_marker
print('V2.2 runner regression tests: PASS')
print('Frozen HEAD:', HEAD)
print('Evidence root:', ROBUST_ROOT)


## 2. Freeze/verify the experiment protocol

In [ ]:
import yaml
cfg = yaml.safe_load((ROOT/'configs'/'reliability_aware_experiments.yaml').read_text())
protocol = {
    'run_id':'core_tfm_jmlr_robustness_v2_2',
    'source_commit_at_freeze':HEAD,
    'inference_engine':'notebooks/CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb',
    'multi_seed':cfg['seed_robustness'],
    'context_size':cfg['context_size'],
    'safe_selective':cfg['safe_selective'],
    'rare_class_sensitivity':cfg['rare_class_sensitivity'],
    'primary_models':cfg['models']['primary'],
    'boundary_models':cfg['models'].get('boundary',[]),
    'outcome_blind_freeze':True,
}
if PROTOCOL_PATH.exists():
    frozen_protocol = json.loads(PROTOCOL_PATH.read_text())
    assert frozen_protocol == protocol, 'V2.2 protocol mismatch: do not mix evidence.'
    print('Existing V2.2 protocol verified.')
else:
    PROTOCOL_PATH.write_text(json.dumps(protocol,indent=2))
    frozen_protocol = protocol
    print('V2.2 protocol frozen.')
display(frozen_protocol)


## 3. Static compile preflight of the real Q1 engine

In [ ]:
TEMPLATE_PATH = ROOT/'notebooks'/'CoRe_TFM_Q1_FAST_COMPLETE_256_Colab.ipynb'
template = load_notebook(TEMPLATE_PATH)
preflight_nb = patch_q1_notebook(
    template, run_id='_static_preflight/seed_11', seed=11,
    train_limit=256, test_limit=128, drive_base=str(ROBUST_ROOT),
    session_minutes=25, shard_minutes=25,
    disable_controlled_replications=True, disable_selection_ablations=True, disable_validation_sensitivity=True,
)
preflight_sources = code_cells_through_shard_12e(preflight_nb)
assert len(preflight_sources) >= 10
print(f'STATIC ENGINE PREFLIGHT PASS: {len(preflight_sources)} Python cells compile through 12E.')


## 4. Select the next incomplete real-TFM variant

In [ ]:
seed_variants=[{'group':'multi_seed','seed':int(s),'train_limit':int(cfg['seed_robustness']['train_limit']),'test_limit':int(cfg['seed_robustness']['test_limit']),'name':f'seed_{s}'} for s in cfg['seed_robustness']['seeds']]
context_variants=[{'group':'context_size','seed':int(s),'train_limit':int(n),'test_limit':int(cfg['context_size'].get('test_limit',128)),'name':f'seed_{s}_train_{n}'} for s in cfg['context_size']['seeds'] for n in cfg['context_size']['train_sizes']]
QUEUE=seed_variants+context_variants
status_rows=[]; NEXT=None
for v in QUEUE:
    st=fold_result_status(ROBUST_ROOT/v['group']/v['name'])
    status_rows.append({**v,'complete':st.get('complete',False),'rows':st.get('rows',0),'fold_cells':st.get('fold_cells',0),'failure_count':st.get('failure_count',0),'reason':st.get('reason')})
    if NEXT is None and not st.get('complete',False): NEXT=v
display(pd.DataFrame(status_rows))
print('NEXT VARIANT:',NEXT)
if NEXT is not None:
    ns=fold_result_status(ROBUST_ROOT/NEXT['group']/NEXT['name'])
    if ns.get('last_failure'):
        print('LAST RECORDED FAILURE:'); print(json.dumps(ns['last_failure'],indent=2))


## 5. Execute/resume one expensive real-TFM variant

In [ ]:
if NEXT is None:
    print('All prespecified multi-seed/context variants are structurally complete.')
else:
    patched=patch_q1_notebook(
        load_notebook(TEMPLATE_PATH), run_id=f"{NEXT['group']}/{NEXT['name']}",
        seed=NEXT['seed'],train_limit=NEXT['train_limit'],test_limit=NEXT['test_limit'],
        drive_base=str(ROBUST_ROOT),session_minutes=25,shard_minutes=25,
        disable_controlled_replications=True,disable_selection_ablations=True,disable_validation_sensitivity=True,
    )
    sources=code_cells_through_shard_12e(patched)
    engine_ns={'__name__':'__robustness_exec__'}
    for idx,source in enumerate(sources,1):
        first=source.lstrip().splitlines()[0] if source.strip() else '<empty>'
        print(f'--- Q1 engine {idx}/{len(sources)}: {first} ---',flush=True)
        try:
            exec(compile(source,f'<q1_engine_cell_{idx}>','exec'),engine_ns,engine_ns)
        except Exception:
            print(f'ENGINE CELL FAILED: {idx} :: {first}'); traceback.print_exc(); raise
    run_dir=ROBUST_ROOT/NEXT['group']/NEXT['name']
    st=fold_result_status(run_dir)
    print(json.dumps(st,indent=2))
    if st.get('complete'):
        print('VARIANT COMPLETE:',write_complete_marker(run_dir,{'group':NEXT['group'],'seed':NEXT['seed'],'requested_train_limit':NEXT['train_limit'],'requested_test_limit':NEXT['test_limit'],'source_commit':HEAD,'inference_engine':'original_Q1_notebook_through_12E'}))
    else:
        print('Variant incomplete; Run all again to resume.')
        if st.get('last_failure'): print(json.dumps(st['last_failure'],indent=2))


## 6. Aggregate progress and rare-class exclusion sensitivity

In [ ]:
thresholds=cfg['rare_class_sensitivity']['minimum_support_thresholds']
subprocess.run([sys.executable,str(ROOT/'experiments'/'summarize_robustness_runs.py'),'--root',str(ROBUST_ROOT),'--seeds',','.join(map(str,cfg['seed_robustness']['seeds'])),'--context-seeds',','.join(map(str,cfg['context_size']['seeds'])),'--context-sizes',','.join(map(str,cfg['context_size']['train_sizes'])),'--rare-thresholds',','.join(map(str,thresholds))],cwd=ROOT,check=True)
progress=json.loads((ROBUST_ROOT/'ROBUSTNESS_STATUS.json').read_text())
print({k:v.get('complete') for k,v in progress.items()})


## 7. Controlled Safe Selective CoRe

In [ ]:
SAFE_DIR=ROBUST_ROOT/'safe_selective'
if not (SAFE_DIR/'COMPLETE.json').exists():
    tmp=ROOT/'results'/'safe_selective_controlled_v1'
    if tmp.exists(): shutil.rmtree(tmp)
    subprocess.run([sys.executable,str(ROOT/'experiments'/'run_safe_selective_controlled.py'),'--tasks','100','--output-dir',str(tmp)],cwd=ROOT,check=True)
    SAFE_DIR.mkdir(parents=True,exist_ok=True)
    for p in tmp.iterdir(): shutil.copy2(p,SAFE_DIR/p.name)
print(json.loads((SAFE_DIR/'COMPLETE.json').read_text()))


## 8. Archive-derived view-reliability proxy

In [ ]:
VIEW_DIR=ROBUST_ROOT/'view_reliability'; VIEW_DIR.mkdir(parents=True,exist_ok=True)
derived=ROOT/'results'/'reliability_aware_v1'
if derived.exists(): shutil.rmtree(derived)
subprocess.run([sys.executable,str(ROOT/'experiments'/'run_reliability_aware_suite.py'),'--fold-results',str(ROOT/'results'/'q1_fast_complete_256_v1'/'fold_results.csv'),'--output',str(derived)],cwd=ROOT,check=True)
for name in ['model_view_reliability_proxy.csv','inconsistency_vs_gain.csv','inconsistency_vs_gain_correlations.json']: shutil.copy2(derived/name,VIEW_DIR/name)
proxy={'complete':True,'scope':'archive_derived_proxy_only','fresh_per_example_direct_marginal_archive':False,'claim_guard':'Diagnostic proxy only; not a fresh per-example TFM reliability rerun.'}
(VIEW_DIR/'PROXY_COMPLETE.json').write_text(json.dumps(proxy,indent=2)); print(proxy)


## 9. Final evidence gate

In [ ]:
def exists(group,name='COMPLETE.json'): return (ROBUST_ROOT/group/name).exists()
core={'multi_seed':exists('multi_seed'),'context_size':exists('context_size'),'rare_class_exclusion':exists('rare_class'),'safe_selective_controlled':exists('safe_selective'),'view_reliability_proxy':exists('view_reliability','PROXY_COMPLETE.json')}
full={**core,'fresh_per_example_view_reliability':exists('view_reliability','FRESH_COMPLETE.json'),'third_tfm':exists('third_tfm')}
print('CORE STATUS:',core); print('CORE ROBUSTNESS GATE:','PASS' if all(core.values()) else 'PENDING')
print('FULL JMLR STATUS:',full); print('FULL JMLR GATE:','PASS' if all(full.values()) else 'PENDING')
